# US-105 Faster-Whisper and Silero VAD benchmark

Feasibility spike only. This notebook does not change the production AI worker.

Before running:

1. Turn on a Kaggle GPU accelerator.
2. Turn on Internet so the notebook can clone the public repository and download model weights. No Kaggle API token and no GitHub write token are required.
3. Paste the full commit SHA below.
4. Attach the audio dataset and set `DATASET` to its `/kaggle/input/...` directory.

Outputs are written under `/kaggle/working`. Download `asr-benchmark.zip` when the last cell finishes.


In [ ]:
BENCHMARK_SHA = ""  # full git commit to measure, for example 9620eb7cf2c2fda2f61af89383dad07107c7960d
DATASET = "/kaggle/input/editagent-asr"
INCLUDE_CPU = False
BUDGET_SECONDS = 3600
CLIP_TIMEOUT_SECONDS = 900
REPO_URL = "https://github.com/mahmoudemad68/Your_Editor.git"
WORK = "/kaggle/working"

if not BENCHMARK_SHA or any(character not in "0123456789abcdef" for character in BENCHMARK_SHA):
    raise SystemExit("Set BENCHMARK_SHA to the full lowercase commit before running.")


Clone the public repository and check out that exact commit.

In [ ]:
import subprocess
from pathlib import Path

work = Path(WORK)
work.mkdir(parents=True, exist_ok=True)
repo = work / "Your_Editor"
if not repo.exists():
    subprocess.check_call(["git", "clone", REPO_URL, str(repo)])
subprocess.check_call(["git", "fetch", "origin", BENCHMARK_SHA], cwd=repo)
subprocess.check_call(["git", "checkout", "--detach", BENCHMARK_SHA], cwd=repo)
head = subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=repo, text=True).strip()
if head != BENCHMARK_SHA:
    raise SystemExit(f"Checked out {head}, expected {BENCHMARK_SHA}.")
print(head)


Install the benchmark dependencies. This step downloads Python packages and, later, model weights.

In [ ]:
import subprocess
import sys

subprocess.check_call(
    [sys.executable, "-m", "pip", "install", "-r", "tools/benchmarks/asr/requirements.txt"],
    cwd=repo,
)


Record the CPU and, when present, the GPU name, memory, driver, and CUDA version.

In [ ]:
import sys

sys.path.insert(0, str(repo / "tools/benchmarks/asr"))
import benchmark

hardware = benchmark.collect_hardware(BENCHMARK_SHA)
probe = benchmark.probe_devices()
print(hardware)
print(probe)


Validate the attached manifest. The run stops when the document is invalid or the audio is under 10 minutes.

In [ ]:
import json
from pathlib import Path

from manifest import load_manifest, validate_manifest

dataset = Path(DATASET)
checked = validate_manifest(load_manifest(dataset / "manifest.json"), dataset, require_files=True)
summary = {key: value for key, value in checked.items() if key != "clips"}
print(json.dumps(summary, indent=2))
if not checked["ok"] or not checked["meets_minimum_total"]:
    raise SystemExit("The dataset manifest is not ready for the benchmark.")


Run the GPU matrix. CPU int8 runs only when INCLUDE_CPU is true. CPU float16 is recorded as unsupported.

In [ ]:
from pathlib import Path

import benchmark

output = Path(WORK) / "asr-results"
benchmark.run_benchmark(
    dataset / "manifest.json",
    dataset,
    output,
    BENCHMARK_SHA,
    include_cpu=INCLUDE_CPU,
    repeats=1,
    warmup=1,
    clip_timeout_seconds=CLIP_TIMEOUT_SECONDS,
    budget_seconds=BUDGET_SECONDS,
    probe=probe,
)
benchmark.run_vad_benchmark(dataset / "manifest.json", dataset, output, BENCHMARK_SHA)
print("runs written", len(list((output / "runs").glob("*.json"))))


Score WER, word timestamps, and Silero intervals. Missing references stay INSUFFICIENT_REFERENCE.

In [ ]:
import json

import evaluate

quality = evaluate.evaluate_manifest(
    dataset / "manifest.json",
    dataset,
    output / "hypotheses",
    output / "vad",
)
(output / "quality.json").write_text(json.dumps(quality, indent=2) + "\n", encoding="utf-8")
print(json.dumps(quality, indent=2))


Build the zip to download from Kaggle. It includes the commit SHA, raw JSON, the CSV, hardware, quality, and errors.

In [ ]:
archive = benchmark.export_bundle(output, Path(WORK) / "asr-benchmark.zip", BENCHMARK_SHA)
print(archive)
